# Research Summarization Engine

An agentic pipeline that takes a single natural-language question, routes it to a specialized "assistant" persona, generates web search queries, scrapes and summarizes the results, and writes a final long-form research report.

**Pipeline overview:**
1. Select a specialized assistant persona for the question (few-shot prompt)
2. Parse the persona/instructions out of the LLM's response
3. Generate several targeted web search queries
4. Run the web searches to get result URLs
5. Scrape the text content of each URL
6. Summarize each scraped page with respect to the original question
7. Merge all summaries into one text block, with source URLs attached
8. Generate the final research report from the merged summaries

## Setup & Imports

- `web_search` / `web_scrape` (`web_searching.py`) — DuckDuckGo search wrapper and a `requests` + `BeautifulSoup` page scraper
- `llm` (`llm_model.py`) — the chat model client used for every LLM call in this notebook
- Prompt templates (`prompts.py`) — one template per pipeline stage
- `parse_llm_json` (`utils.py`) — extracts and parses a JSON object/array out of an LLM's free-text response (LLMs return text, not real JSON, so this bridges the gap for every stage that needs structured output)

In [ ]:
import json
import re
import sys
sys.path.append("..")

from web_searching import web_search, web_scrape
from llm_model import llm 
from prompts import (
    ASSISTANT_SELECTION_PROMPT_TEMPLATE,
    WEB_SEARCH_PROMPT_TEMPLATE,
    SUMMARY_PROMPT_TEMPLATE,
    RESEARCH_REPORT_PROMPT_TEMPLATE
)
from utils import parse_llm_json

## Configuration

- `NUM_SEARCH_QUERIES` / `NUM_SEARCH_RESULTS_PER_QUERY` — control how many search queries the LLM should generate and how many result URLs to fetch per query (expected total URLs = product of the two)
- `RESULT_TEXT_MAX_CHARACTERS` — truncates each scraped page's text before it's sent to the LLM, to keep prompts within a reasonable size
- `question` — the single research question that drives the whole pipeline

In [2]:
NUM_SEARCH_QUERIES = 2
NUM_SEARCH_RESULTS_PER_QUERY = 3
RESULT_TEXT_MAX_CHARACTERS = 10000
question = 'What can I see and do in the Spanish town of Astorga?'


## Step 1: Select a Specialized Assistant

`ASSISTANT_SELECTION_PROMPT_TEMPLATE` (`prompts.py`) is a few-shot prompt: it gives the LLM three worked examples (finance, tour guide, sports), each showing the exact `{assistant_type, assistant_instructions, user_question}` JSON shape, then asks it to produce the same structure for the real question. This lets the LLM pick a persona and matching system-style instructions tailored to the topic, instead of answering with generic, unfocused prose.

In [4]:
assistant_selection_prompt = ASSISTANT_SELECTION_PROMPT_TEMPLATE.format(user_question=question)
assistant_instructions = llm.invoke(assistant_selection_prompt)

## Parse the Assistant Selection Response

`llm.invoke(...)` returns a chat message whose `.content` is plain text — even when the prompt asks for JSON, the model doesn't return a real object, just text that *looks like* JSON (often wrapped in extra prose or markdown fences). `parse_llm_json` (`utils.py`) extracts the JSON substring with a regex and parses it into a real Python `dict`, so the next steps can index into it (`assistant_instructions_dict['assistant_instructions']`, etc.) instead of treating it as one opaque string.

In [ ]:
assistant_instructions_dict =  parse_llm_json(assistant_instructions.content)
print(assistant_instructions_dict)

## Step 2: Generate Web Search Queries

`WEB_SEARCH_PROMPT_TEMPLATE` (`prompts.py`) is filled in with the persona's instructions, `NUM_SEARCH_QUERIES`, and the original question. The LLM (acting as that persona) breaks the one natural-language question down into several more targeted, keyword-style search queries and returns them as a JSON list — since a single broad question is often too vague for good search-engine results.

> **Known issue:** the prompt's example JSON is hardcoded to show 3 queries (`query1, query2, query3`) regardless of `num_search_queries`. The LLM tends to pattern-match the concrete example over the parameterized count in the instruction text, so it can return 3 queries even when `NUM_SEARCH_QUERIES = 2`. Worth fixing the template to make the example length dynamic if an exact count matters.

In [6]:
web_search_prompt = WEB_SEARCH_PROMPT_TEMPLATE.format(
    assistant_instructions=assistant_instructions_dict['assistant_instructions'],
    num_search_queries=NUM_SEARCH_QUERIES,
    user_question=assistant_instructions_dict['user_question'])
web_search_queries = llm.invoke(web_search_prompt)
web_search_queries_list = parse_llm_json(web_search_queries.content)
print(web_search_queries_list)

[{'search_query': 'Astorga Spain attractions things to do tourist sites', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}, {'search_query': 'Astorga León history landmarks museums Camino de Santiago', 'user_question': 'What can I see and do in the Spanish town of Astorga?'}]


## Step 3: Run the Web Searches

For each search query, `web_search` (DuckDuckGo) fetches `NUM_SEARCH_RESULTS_PER_QUERY` result URLs. The output is a list of `{result_urls, search_query}` dicts — one entry per query, each holding the URLs found for it. Expected total URL count is `NUM_SEARCH_QUERIES × NUM_SEARCH_RESULTS_PER_QUERY`, though DuckDuckGo's unofficial API can be flaky (rate limiting, occasionally irrelevant results), which can push the actual count off from that.

In [8]:
searches_and_result_urls = [
    {
        'result_urls': web_search(
            web_query=wq['search_query'],
            num_results=NUM_SEARCH_RESULTS_PER_QUERY),
        'search_query': wq['search_query']
    }
    for wq in web_search_queries_list
]
print(searches_and_result_urls)

Impersonate 'chrome_126' does not exist, using 'random'


[{'result_urls': ['https://en.wikipedia.org/wiki/Astorga,_Spain', 'https://www.tripadvisor.com/Tourism-g668532-Astorga_Province_of_Leon_Castile_and_Leon-Vacations.html', 'https://www.spain.info/en/destination/astorga/'], 'search_query': 'Astorga Spain attractions things to do tourist sites'}, {'result_urls': ['https://www.bing.com/aclick?ld=e8A9VXe0gHH6eKzjjUYsiSRjVUCUw6O0QJ1IVX2Gm60YcwIQ2ObJ0EWQCcpNY9A4W3O3RLMbq7TqrZVjSUAG8c8uYnHrb0csYPHmztAXOulREd1fHxlioPziHTDB9jcuz3-Kp1AaCkSylrTMSW_XEdoMp48EPa6tYXEKjRNXmxIswfNEj_-C396GEVEQh8lgQB03B0aw&u=aHR0cHMlM2ElMmYlMmZzYW50aWFnb3dheXMuY29tJTJmZGUlM2Zjb2RlcHJvbW8lM2RERV9BTF9TRUFSQ0hfQiUyNm1zY2xraWQlM2Q0YWZlZWIzZTMzODcxNGJjZDkzMjgxYTIwYjYwNmY0NiUyNnV0bV9zb3VyY2UlM2RiaW5nJTI2dXRtX21lZGl1bSUzZGNwYyUyNnV0bV9jYW1wYWlnbiUzZERFX1NFQVJDSF9CSU5HJTI2dXRtX3Rlcm0lM2RKYWtvYnN3ZWclMjZ1dG1fY29udGVudCUzZEpha29ic3dlZw&rlid=4afeeb3e338714bcd93281a20b606f46', 'https://astorga.co/en/', 'https://www.nomads-travel-guide.com/city/astorga/'], 'search_query': 'Astorga Le

## Step 4: Scrape Each Result Page

Flattens the nested `searches_and_result_urls` structure into one flat list — one entry per URL — and scrapes each page's visible text with `web_scrape` (`requests` + `BeautifulSoup`), truncated to `RESULT_TEXT_MAX_CHARACTERS`. `web_scrape` never raises: on a failed request (e.g. a 403) it returns an error string as the "page text" instead, so every URL still produces an entry here even when the fetch failed — the failure just surfaces later as an unhelpful summary rather than a crash.

In [9]:
result_text_list = [{
    'result_text': web_scrape(url=url)[:RESULT_TEXT_MAX_CHARACTERS],
    'result_url': url,
    'search_query': item['search_query']}
    for item in searches_and_result_urls
    for url in item['result_urls']]

## Step 5: Summarize Each Source

For every scraped page, `SUMMARY_PROMPT_TEMPLATE` asks the LLM to answer the original search query using that page's text (or just summarize it if the text doesn't answer the question). Each `llm.invoke()` call is wrapped in `try/except` — if one call fails (rate limit, timeout, provider error), that URL is skipped and logged instead of aborting the whole batch and silently truncating `result_text_summary_list` to whatever succeeded before the failure.

In [13]:
result_text_summary_list = []
for rt in result_text_list:
    summary_prompt = SUMMARY_PROMPT_TEMPLATE.format(
        search_result_text=rt['result_text'],
        search_query=rt['search_query'])

    try:
        text_summary = llm.invoke(summary_prompt)
    except Exception as e:
        print(f"Skipping {rt['result_url']}: {e}")
        continue

    result_text_summary_list.append({
        'text_summary': text_summary.content,
        'result_url': rt['result_url'],
        'search_query': rt['search_query']})

print(f'{len(result_text_summary_list)}/{len(result_text_list)} summaries completed')

6/6 summaries completed


## Step 6: Merge Summaries into One Text Block

LLM prompts take plain text, not Python lists/dicts, so this serializes each `{result_url, text_summary}` dict into a `"Source URL: ...\nSummary: ..."` block (keeping the URL attached so the final report can cite sources) and joins them all into one string, `appended_result_summaries`, for the final prompt.

> Note: if you print a very long string like this directly, Jupyter/VS Code's output panel can silently truncate it — loop and print each block separately, or write it to a file, if you need to inspect the full thing.

In [14]:
stringified_summary_list = [
    f'Source URL: {sr["result_url"]}\nSummary: {sr["text_summary"]}' 
        for sr in result_text_summary_list]
appended_result_summaries = '\n'.join(stringified_summary_list)

## Step 7: Generate the Final Research Report

`RESEARCH_REPORT_PROMPT_TEMPLATE` (`prompts.py`) combines all the merged source summaries with the original question and asks the LLM to write a long-form (1,200+ word), structured, APA-formatted report with a clear opinion and cited source URLs — the final deliverable of the pipeline.

In [ ]:
research_report_prompt = RESEARCH_REPORT_PROMPT_TEMPLATE.format(
    research_summary=appended_result_summaries,
    user_question=question
)
research_report = llm.invoke(research_report_prompt)

print(f'strigified_summary_list={stringified_summary_list}')
print(f'merged_result_summaries={appended_result_summaries}')
print(f'research_report={research_report}')